In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import SGDRegressor, Ridge
from sklearn.model_selection import KFold, StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error as mse, r2_score 
from sklearn.metrics import mean_absolute_error as mae
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
import plotly.express as px
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency
from scipy.stats import kruskal
import xgboost as xgb
from xgboost import XGBRegressor

In [ ]:
import warnings 
warnings.filterwarnings('ignore')

In [ ]:
#Supervised regression 
hosp = pd.read_csv('/kaggle/input/datasets/prathyushasista/health-insurance-dataset/Hospitalisation details.csv')

In [ ]:
med = pd.read_csv('/kaggle/input/datasets/prathyushasista/health-insurance-dataset/Medical Examinations.csv')
names = pd.read_excel('/kaggle/input/datasets/prathyushasista/health-insurance-dataset/Names.xlsx')

In [ ]:
hosp.info()

In [ ]:
med.info()

In [ ]:
names.info()

In [ ]:
# Collate the files so that all the information is in one place using the common column - customer id
master_data = pd.merge(hosp, med, how = "inner", on = 'Customer ID')

In [ ]:
master_data = master_data.merge(names, on = "Customer ID")

In [ ]:
master_data.info()

In [ ]:
master_data.head()

In [ ]:
master_data.columns = master_data.columns.str.replace(" ", "_").str.lower()

In [ ]:
master_data.columns

In [ ]:
#Check for missing values in the dataset
master_data[:50]

In [ ]:
#Check for missing data - Since no data is missing we don't need to perform any missing value imputation
master_data.isnull().sum()

In [ ]:
#Find the percentage of rows that have trivial value and delete such rows if they do not contain significant information
(master_data == '?').sum()

In [ ]:
#Row wise percentage of special character data
missing_percent = (master_data == '?').sum(axis=1)/master_data.shape[1]*100
missing_percent

In [ ]:
missing_percent[missing_percent > 0]

In [ ]:
#Column wise percentage of special character data
missing_percent_col = (master_data == '?').sum(axis=0)/master_data.shape[1]*100
missing_percent_col[missing_percent_col > 0]

In [ ]:
data = master_data.drop(index = missing_percent[missing_percent > 0].index)

In [ ]:
data.shape

In [ ]:
master_data.shape

In [ ]:
master_data.head()

In [ ]:
#Use the necessary transformation methods to deal with the nominal and ordinal categorical variables in the dataset
#Ordinal Categorical = hospital_tier, city_tier
label_encoder = LabelEncoder()
data['hospital_tier']= label_encoder.fit_transform(data['hospital_tier'])
data['city_tier']= label_encoder.fit_transform(data['city_tier'])
data['month']= label_encoder.fit_transform(data['month'])
data['heart_issues']= label_encoder.fit_transform(data['heart_issues'])
data['any_transplants']= label_encoder.fit_transform(data['any_transplants'])
data['cancer_history']= label_encoder.fit_transform(data['cancer_history'])
data['smoker']= label_encoder.fit_transform(data['smoker'])
encoded_data = data.copy()
encoded_data

In [ ]:
#Design a suitable strategy to create dummy variables with restraints for state ids R1013, R1011, R1012
encoded_data.state_id.value_counts()

In [ ]:
encoded_data.state_id.nunique()

In [ ]:
dummies = pd.get_dummies(encoded_data['state_id'], prefix="state_id")
selected_states = dummies[['state_id_R1011', 'state_id_R1012','state_id_R1013']] 
selected_states

In [ ]:
#Nominal encoding of state_id
merged_data = pd.concat([encoded_data, selected_states], axis=1) 
merged_data.drop('state_id', axis='columns', inplace=True) 
merged_data.head()

In [ ]:
#The variable NumberOfMajorSurgeries also appears to have string values.Apply a suitable method to clean up this variable.
merged_data.numberofmajorsurgeries.value_counts()

In [ ]:
merged_data.numberofmajorsurgeries.unique()

In [ ]:
merged_data.numberofmajorsurgeries.replace('No major surgery', 0, inplace=True)

In [ ]:
merged_data.numberofmajorsurgeries.isnull().sum()

In [ ]:
# Handled missing values by replacing with a 0
merged_data.numberofmajorsurgeries.replace(np.nan, 0, inplace=True)

In [ ]:
merged_data.numberofmajorsurgeries = merged_data.numberofmajorsurgeries.astype(int)

In [ ]:
merged_data.dtypes

In [ ]:
merged_data.state_id_R1011 = merged_data.state_id_R1011.astype(int)
merged_data.state_id_R1012 = merged_data.state_id_R1012.astype(int)
merged_data.state_id_R1013 = merged_data.state_id_R1013.astype(int)

In [ ]:
merged_data.dtypes

In [ ]:
merged_data[:50]

In [ ]:
#Age appears to be a significant factor in this analysis. Calculate the patients' ages based on their dates of birth.
merged_data.year.isnull().sum()

In [ ]:
merged_data.dropna(inplace=True)

In [ ]:
merged_data.isnull().sum()

In [ ]:
merged_data.head()

In [ ]:
merged_data.year = merged_data.year.astype(int)

In [ ]:
current_year = datetime.date.today().year
current_year

In [ ]:
merged_data["Age"] = current_year - merged_data.year

In [ ]:
merged_data.head()

In [ ]:
#Calculate the gender of the patient based on the salutation used in the name␣ ↪column
merged_data.name

In [ ]:
merged_data['title'] = merged_data.name.str.split('[,.]'). str[1].str.strip()

In [ ]:
merged_data['title'].isnull().sum()

In [ ]:
merged_data['title'].value_counts()

In [ ]:
merged_data.info()

In [ ]:
merged_data['title'] = np.where(merged_data['title'].str.strip() == 'Mr', 'M', 'F')

In [ ]:
merged_data["title"].value_counts()

In [ ]:
merged_data.rename(columns = {'title':'Gender'}, inplace = True)

In [ ]:
merged_data["Gender"].value_counts()

In [ ]:
male_data = merged_data[merged_data.Gender == 'M']
male_data.shape
female_data = merged_data[merged_data.Gender == 'F']
female_data.charges

In [ ]:
#Visualize the distribution of costs using a histogram, box and whisker plot, and swarm plot.
#State how the distribution is different across gender and tiers of hospitals
#Histogram
male_data = merged_data[merged_data.Gender == 'M']
female_data = merged_data[merged_data.Gender == 'F']
plt.hist([male_data.charges, female_data.charges], bins=20, stacked=True, color=['cyan', 'Purple'], edgecolor='black')

# Adding labels and title
plt.xlabel('Cost of hospitalisation')
plt.ylabel('Frequency')
plt.title('Cost of hospitalisation by gender')
plt.legend(['Male', 'Female'])

# Display the plot
plt.show()

In [ ]:
#Gender based hospital charges based on hospital tier
#box and whisker plot
sns.set(style="whitegrid")
sns.boxplot(x=merged_data.hospital_tier, y=merged_data.charges,
    hue=merged_data.Gender,
data=merged_data, palette="Set2", dodge=True)

In [ ]:
#Swarm plot showing the distribution of hospitalization costs for each hospital tier for each gender
sns.swarmplot(x=merged_data.hospital_tier,
                  y=merged_data.charges,
              hue=merged_data.Gender,
              data=merged_data)
plt.show()

In [ ]:
#Radar chart to showcase the median hospitalization cost for each tier of hospitals
r1 = merged_data[merged_data['hospital_tier'] == 0].charges.median()
r2 = merged_data[merged_data['hospital_tier'] == 1].charges.median()
r3 = merged_data[merged_data['hospital_tier'] == 2].charges.median()
radar_df = pd.DataFrame(dict(r=[r1, r2, r3], theta=['Tier 0 hospital', 'Tier 1 hospital', 'Tier 2 hospital']))
fig = px.line_polar(radar_df, r= 'r', theta = 'theta', line_close=True) 
fig.show()

In [ ]:
#Frequency table and a stacked bar chart to visualize the count of people in the different tiers of cities and hospitals
#Frequency Table
pd.crosstab(merged_data.city_tier, merged_data.hospital_tier)

In [ ]:
#No. of patients in each tier by city
for c in merged_data.city_tier.unique():
    city_data= len(merged_data[merged_data.city_tier == c])

#No. of patients in each tier by hospital
for c in merged_data.hospital_tier.unique():
    hospital_data= len(merged_data[merged_data.hospital_tier == c])

In [ ]:
#No. of patients in each tier by city and by hospital
label = merged_data.hospital_tier
fig = plt.figure(figsize = (10, 7))
city_tier = np.array(city_data)
hospital_tier = np.array(hospital_data)
plt.xticks(np.arange(3), ('Tier-1', 'Tier-2', 'Tier-3'))
plt.bar(label, city_tier, color='g', width=0.3)
plt.bar(label, hospital_tier, bottom=city_tier, color='y', width=0.3)
plt.xlabel("Tiers")
plt.ylabel("No. of patients")
plt.title("No. of patients in each city and hospital tiers")
plt.show()

In [ ]:
#Null Hypothesis testing
#The average hospitalization costs for the three types of hospitals are not␣ ↪significantly different.
#ANNOVA - pick charges, hospital tier
#H0 - Cost is not different
#HA - Cost is different for each tier
#Check p value for testing the hypothesis
anova_data = [merged_data['charges'], merged_data['hospital_tier']] 
stat, p = f_oneway(*anova_data)
print("stat=%.3f, p=%.3f" % (stat, p)) 
if p > 0.05:
    print('Hospitalization costs are independent on the tier of the hospital') 
else:
    print('Hospitalization costs vary based on the tier of the hospital')

#Since p<0.05, we can REJECT the null hypothesis that the average hospitalization costs for the three types of hospitals are not significantly different.

In [ ]:
#The average hospitalization costs for the three types of cities are not significantly different.
#ANNOVA - pick charges, city tier
#H0 - Cost is not different
#HA - Cost is different for each tier
#Check p value for testing the hypothesis
anova_data = [merged_data['charges'], merged_data['city_tier']] 
stat, p = f_oneway(*anova_data)
print("stat=%.3f, p=%.3f" % (stat, p)) 
if p > 0.05:
    print('Hospitalization costs are independent on the tier of the city') 
else:
    print('Hospitalization costs vary based on the tier of the city') 
    #Since p<0.05, we can REJECT the null hypothesis that the average hospitalization costs for the three types of cities are not significantly different.

In [ ]:
non_smoker_data = merged_data[merged_data['smoker'] == 0].charges.median()
smoker_data = merged_data[merged_data['smoker'] == 1].charges.median()
smoker_data, non_smoker_data
stat, p = kruskal(smoker_data, non_smoker_data)
print("stat=%.3f, p=%.3f" % (stat, p)) 
if p > 0.05:
    print('Hospitalization costs are independent of the smoking status') 
else:
    print('Hospitalization costs vary based on the smoking status')

In [ ]:
#Smoking and heart issues are independent.
#Perform Chi-squared test
#H0 - Smoking and heart issues are independent
#HA - Smoking and heart issues are dependent
chi_sq_data = [[merged_data['smoker'].value_counts(), merged_data['heart_issues'].value_counts()]]
stat, p, dof, expected = chi2_contingency(chi_sq_data) 
alpha = 0.05
print("stat=%.3f, p=%.3f" % (stat, p))
if p > 0.05:
    print('Smoking and heart issues are independent') 
else:
    print('Smoking and heart issues are dependent')
#Since p>0.05, we ACCEPT the null hypothesis that Smoking and heart issues are independent

In [ ]:
#Encoding the Gender column
merged_data['Gender'] = label_encoder.fit_transform(merged_data['Gender'])

In [ ]:
#Examine the correlation between predictors to identify highly correlated␣ ↪predictors
corr_data = merged_data.copy()
corr_data.drop(['year', 'date', 'month', 'name', 'customer_id'], axis=1, inplace=True) 
corr_data.head()

In [ ]:
fig = plt.figure(figsize = (30, 20))
sns.heatmap(corr_data.corr(), cmap='YlGnBu', linewidths=2, linecolor='white', annot=True, cbar=True)
plt.show()
#Age and any transplants, hba1c are highly correlated
# Heart issues and number of majoir surgeries are correlated
#Since Gender and smoker data has very correlation with any of the other factors, we can remove them
#corr_data.drop(['Gender_F', 'Gender_M', 'smoker_yes', 'smoker_No'], axis=1, ␣ ↪inplace=True)
#corr_data.drop(['bmi'], axis=1,  inplace=True)

In [ ]:
#ML model creation
ml_data = merged_data.copy()
ml_data.drop(['name', 'customer_id', 'year', 'date', 'month'], axis=1, inplace=True)
X = ml_data.drop('charges', axis=1) 
y = ml_data['charges']

#Train test split for model building
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)
X_train.shape, X_test.shape

In [ ]:
#Incorporate sklearn-pipelines to streamline the workflow
n_splits = 5
pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('regressor', Ridge())])

#parameter for hyperparameter tuning
parameters = {'regressor__alpha':[0.001, 0.01,0.1,1,10,100]} 

#kfold
kfold = KFold(n_splits = n_splits, shuffle=True, random_state=42) 

#Grid search
model_ridge = GridSearchCV(pipeline, parameters, cv=kfold, scoring="neg_mean_absolute_error", verbose=True, refit=True, return_train_score=True)

In [ ]:
model_ridge.fit(X_train,y_train) 


In [ ]:
model_ridge.best_params_

In [ ]:
model_ridge.best_estimator_

In [ ]:
model_ridge.best_score_

In [ ]:
model_ridge.feature_names_in_

In [ ]:
grid_search_model = model_ridge.best_estimator_
grid_csv_model =  Ridge(alpha= 0.001)
grid_csv_model.fit(X_train, y_train)
# Print classification report for the Grid Search model
#print("Grid Search - Classification Report:")
#print(classification_report(y_test, y_pred_grid))

In [ ]:
grid_csv_model.score(X_test, y_test)

In [ ]:
y_pred = grid_csv_model.predict(X_test)

In [ ]:
ridge_mse = mse(y_test, y_pred)
ridge_r2 = r2_score(y_test, y_pred)
ridge_mae = mae(y_test, y_pred)
print("Ridge Mean squared error: ", ridge_mse)
print("Ridge Mean Absolute error: ", ridge_mae)
print("Ridge R2 score: ", ridge_r2)

In [ ]:
grid_csv_model.n_features_in_

In [ ]:
feature_importance = grid_csv_model.coef_

In [ ]:
grid_csv_model.feature_names_in_

In [ ]:
feature_table = pd.DataFrame(feature_importance, index=X.columns, columns=['Feature_importance'])
feature_table.sort_values(by = 'Feature_importance', ascending=False)

In [ ]:
# Develop Gradient Boost model and determine the variable importance scores and identify the redundant variables
rfg = RandomForestRegressor(n_estimators = 1000, random_state=42)
rfg.fit(X_train, y_train)

In [ ]:
rfg.score(X_test, y_test)

In [ ]:
y_pred_rfg = rfg.predict(X_test)

In [ ]:
rfg_mae = mae(y_test, y_pred_rfg)
rfg_mae

In [ ]:
# Gradient Boosting Regressor
params = {
    "n_estimators": 1000,
    "loss": "squared_error",
    "random_state": 42
}
gbr = GradientBoostingRegressor(**params)

In [ ]:
gbr.fit(X_train, y_train)

In [ ]:
gbr.score(X_test, y_test)

In [ ]:
y_pred_gbr = gbr.predict(X_test)

In [ ]:
gbr_mae = mae(y_test, y_pred_gbr)
gbr_mae

#Case scenario: Testing the accuracy of predictions using new data
#Estimate the cost of hospitalization for Christopher, Ms. Jayna (Date of birth 12/28/1988; height 170 cm; and weight 85 kgs).
#She lives with her partner and two children in a tier-1 city, and her state’s State ID is R1011.
#She was found to be nondiabetic (HbA1c = 5.8). She smokes but is otherwise healthy.
#She has had no transplants or major surgeries. Her father died of lung cancer. #Hospitalization costs will be estimated using tier-1 hospitals.

#Calculate age
dt = str(19881228)
calc_date = pd.to_datetime(dt, format="%Y%m%d") 
calc_date

today = datetime.datetime.now()
diff = today - calc_date

age = int(diff.days/365) 
age
#age = 36

#Calculate BMI:
height = pow(170/100, 2)
weight = 85
height

bmi = weight/height 
bmi
#bmi - 29.41
#Gender - Female - 0

X_train.columns

X_train.hospital_tier.value_counts()

data = [[2,0,0,29.41,5.8,0,0,1,0,1,1,0,0,36,0]]
new_data = pd.DataFrame(data, columns=['children', 'hospital_tier','city_tier', 'bmi', 'hba1c',
'heart_issues', 'any_transplants', 'cancer_history', 'numberofmajorsurgeries', 'smoker', 'state_id_R1011', 'state_id_R1012', 'state_id_R1013', 'Age', 'Gender'])
new_data

#Find the predicted hospitalization cost using the best models
hospital_cost = []
#Predicted cost using Ridge Regression model
cost = grid_csv_model.predict(new_data)
hospital_cost.append(cost)

#Predicted cost using Random Forest Regression model
cost_rfg = rfg.predict(new_data)
hospital_cost.append(cost_rfg)

#Predicted cost using Gradient Boost Regression model
cost_gbr = gbr.predict(new_data)
hospital_cost.append(cost_gbr)

hospital_cost

#Average predicted cost
Avg_cost = np.mean(hospital_cost)
print("The average hospitalization cost for the given case scenario using the best models: ", np.round(Avg_cost, 2))

In [ ]:
import pickle
from sklearn.utils.validation import check_is_fitted

os.makedirs("models", exist_ok=True)

ridge_best = model_ridge.best_estimator_

for name, model in [("RF", model_ridge), ("GB", gbr), ("Ridge", rfg)]:
    try:
        check_is_fitted(model)
        print(f"{name}: fitted")
    except Exception as e:
        print(f"{name}: NOT fitted — {e}")
        
with open("models/ridge.pkl", "wb") as f:
    pickle.dump(ridge_best, f)

with open("models/random_forest.pkl", "wb") as f:
    pickle.dump(rfg, f)

with open("models/gradient_boost.pkl", "wb") as f:
    pickle.dump(gbr, f)

with open("models/preprocessor.pkl", "wb") as f:
    pickle.dump(pipeline, f)

with open("models/feature_names.pkl", "wb") as f:
    pickle.dump(grid_csv_model.feature_names_in_, f)


In [ ]:
X_train.to_pickle("models/X_train.pkl")

In [ ]:
print(ridge_best.named_steps)

ridge_model_only = ridge_best.named_steps["regressor"] 

In [ ]:
preprocessor = ridge_best.named_steps["scaler"]

# Feature names (post-transformation)
feature_names = preprocessor.get_feature_names_out()
print(feature_names)

In [ ]:
import shap

# Tree-based models
rf_explainer = shap.TreeExplainer(rfg)
gb_explainer = shap.TreeExplainer(gbr)

# Linear model — needs background data
X_train_transformed = ridge_best.named_steps["scaler"].transform(X_train)

ridge_explainer = shap.LinearExplainer(ridge_model_only, X_train_transformed)

# Baseline average prediction (used later in the LLM prompt)
base_value = (
    rf_explainer.expected_value +
    gb_explainer.expected_value +
    ridge_explainer.expected_value
) / 3


In [ ]:
new_patient = pd.DataFrame([{
    "children": 2,
    "hospital_tier": 0,
    "city_tier": 0,
    "bmi": 31.4,
    "hba1c": 7,
    "heart_issues": 0,
    "any_transplants": 0,
    "cancer_history": 0,
    "numberofmajorsurgeries": 2,
    "smoker": 0,
    "state_id_R1011": 1,
    "state_id_R1012": 0,
    "state_id_R1013": 0,
    "Age": 36,
    "Gender": 1
}])

#'children' 'hospital_tier' 'city_tier' 'bmi' 'hba1c' 'heart_issues''any_transplants' 'cancer_history' 'numberofmajorsurgeries' 'smoker'
 #'state_id_R1011' 'state_id_R1012' 'state_id_R1013' 'Age' 'Gender'


new_patient_df = pd.DataFrame(new_patient, columns=['children', 'hospital_tier','city_tier', 'bmi', 'hba1c',
'heart_issues', 'any_transplants', 'cancer_history', 'numberofmajorsurgeries', 'smoker', 'state_id_R1011', 'state_id_R1012', 'state_id_R1013', 'Age', 'Gender'])
print(new_patient_df)


In [ ]:
# Compute SHAP values for a sample 
rf_shap_values = rf_explainer.shap_values(new_patient_df)
gb_shap_values = gb_explainer.shap_values(new_patient_df)
ridge_shap_values = ridge_explainer.shap_values(new_patient_df)

# Combine — matches your ensemble's (RF + GB + Ridge) / 3
combined_shap_values = (rf_shap_values + gb_shap_values + ridge_shap_values) / 3


In [ ]:
def to_scalar(x):
    if isinstance(x, np.ndarray):
        return float(x.flatten()[0]) if x.size == 1 else float(x.mean())
    return float(x)

In [ ]:
def explain_patient(new_patient_df, top_n=10):
    # 1. Preprocess the new patient (transform only, never fit)
    patient_transformed = preprocessor.transform(new_patient_df)
    if hasattr(patient_transformed, "toarray"):
        patient_transformed = patient_transformed.toarray()

    # 2. Predictions from each model
    rf_pred = rfg.predict(patient_transformed)[0]
    gb_pred = gbr.predict(patient_transformed)[0]
    ridge_pred = ridge_model_only.predict(patient_transformed)[0]
    final_prediction = (rf_pred + gb_pred + ridge_pred) / 3

    # 3. SHAP values for THIS patient
    rf_shap = rf_explainer.shap_values(patient_transformed)[0]
    gb_shap = gb_explainer.shap_values(patient_transformed)[0]
    ridge_shap = ridge_explainer.shap_values(patient_transformed)[0]
    combined_shap = (rf_shap + gb_shap + ridge_shap) / 3

    # 4. Contribution table
    df = pd.DataFrame({
        "feature": feature_names,
        "shap_contribution": combined_shap
    })
    df["abs_contribution"] = df["shap_contribution"].abs()
    df = df.sort_values("abs_contribution", ascending=False).drop(columns="abs_contribution").head(top_n)

    return final_prediction, df.reset_index(drop=True), new_patient_df.iloc[0].to_dict()

In [ ]:
def build_shap_summary_text(contribution_table, raw_patient_dict):
    lines = []
    for _, row in contribution_table.iterrows():
        # Clean up feature name (remove ColumnTransformer prefixes like 'num__', 'cat__')
        clean_name = row["feature"].split("__")[-1]
        direction = "increased" if row["shap_contribution"] > 0 else "decreased"
        lines.append(f"- {clean_name}: {direction} the estimated cost by ${abs(row['shap_contribution']):.2f}")
    
    patient_info = ", ".join([f"{k}: {v}" for k, v in raw_patient_dict.items()])
    shap_summary = "\n".join(lines)
    
    return patient_info, shap_summary

In [ ]:
def build_llm_prompt(predicted_cost, base_value, patient_info, shap_summary):
    prompt = f"""You are a medical cost analyst explaining a hospitalization cost prediction to a patient in plain, non-technical language.

Patient details: {patient_info}

Baseline average predicted cost: ${base_value:.2f}
Final predicted cost for this patient: ${predicted_cost:.2f}

The following factors influenced this patient's predicted cost, based on a machine learning model:
{shap_summary}

Write a short, clear explanation (3-5 sentences) for the patient describing:
1. What their estimated hospitalization cost is
2. The top 2-3 factors that most influenced this estimate, in plain English
3. Whether each factor increased or decreased their cost, and briefly why that makes sense medically

Avoid technical jargon like "SHAP values" or "feature contribution" — write as if explaining to someone with no data science background. Do not give medical advice. Add a disclaimer at the end saying this is not actual medical advice. Please refer to a doctor for an accurate diagnosis"""
    
    return prompt

In [ ]:
def generate_full_report(new_patient_df, top_n=10):
    """
    Takes raw patient input (DataFrame, one row) and returns:
    - predicted_cost (float)
    - contribution_table (DataFrame, top SHAP features)
    - report_text (str, LLM-generated explanation)
    """
    predicted_cost, contribution_table, raw_patient_dict = explain_patient(new_patient_df, top_n=top_n)
    patient_info, shap_summary = build_shap_summary_text(contribution_table, raw_patient_dict)
    prompt = build_llm_prompt(to_scalar(predicted_cost), to_scalar(base_value), patient_info, shap_summary)
    report_text = generate_report(prompt)

    return predicted_cost, contribution_table, report_text

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN= UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ Hugging Faces API key setup complete.")
except Exception as e:
    print(
        f"🔑 Auth Error: Please make sure you have added 'HF_API_KEY' to your Kaggle secrets. Details: {e}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # loads HF_TOKEN from a .env file automatically

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(model=model_id)  # reads HF_TOKEN from env

from huggingface_hub import notebook_login

def generate_report(prompt, max_tokens=500):
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

In [ ]:
predicted_cost, contribution_table, report_text = generate_full_report(new_patient)
print("\nCost:", predicted_cost)
print(report_text)